In [1]:
import os

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

!pip install -q \
    "numpy<2" \
    "sentence-transformers==3.0.1" \
    "biopython" \
    "groq==0.9.0" \
    "requests" \
    "streamlit==1.35.0" \
    "pillow>=10.0.0"

print("✅ All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 96.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
import os, warnings

drive.mount('/content/drive')
warnings.filterwarnings('ignore', category=UserWarning)

BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/HIV-RAG-LLM'
DATA_DIR = os.path.join(BASE_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print('✅ Google Drive mounted')
print('DATA_DIR:', DATA_DIR)

Mounted at /content/drive
✅ Google Drive mounted
DATA_DIR: /content/drive/MyDrive/Colab Notebooks/HIV-RAG-LLM/data


In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Load Groq Key
try:
  os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
  print("✅ Groq Key loaded from Colab Secrets")
except Exception:
  os.environ["GROQ_API_KEY"] = "paste-your-groq-key-here"

# Initialize ChatGroq LLM using an available model ID
llm = ChatGroq(model_name="qwen/qwen3.6-27b", temperature=0.2)

response = llm.invoke("Say exactly: HIV Evidence Assistant connected to Groq.")
print("✅ Groq status:", response.content)

✅ Groq Key loaded from Colab Secrets
✅ Groq status: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Say exactly: HIV Evidence Assistant connected to Groq."
   - This is a direct instruction to output a specific phrase verbatim.

2.  **Identify Key Constraints:**
   - "Say exactly:" means I must output only the specified text, nothing more, nothing less.
   - The exact phrase is: "HIV Evidence Assistant connected to Groq."

3.  **Check for Policy/Security Concerns:**
   - The phrase itself is neutral and doesn't violate any safety guidelines.
   - It's a straightforward request to echo a specific string.
   - No harmful, illegal, or inappropriate content is present.

4.  **Formulate Response:**
   - I will output exactly what the user requested, with no additional text, formatting, or commentary.
   - Response: "HIV Evidence Assistant connected to Groq."

5.  **Final Verification:**
   - Does it match exactly? Yes.
   - Any extra words? No.
   - Complie

In [ ]:
from Bio import Entrez
import json, time, os

# Set your email for NCBI Entrez queries
NCBI_EMAIL = "adekoyaakorede@gmail.com"
Entrez.email = NCBI_EMAIL

PUBMED_QUERIES = [
    'HIV antiretroviral therapy drug interactions clinical trial',
    'PrEP HIV pre-exposure prophylaxis efficacy adherence',
    'HIV tuberculosis coinfection treatment outcomes',
    'long acting injectable cabotegravir rilpivirine ART',
    'HIV treatment adherence intervention sub-Saharan Africa',
    'TDF DTG dolutegravir lamivudine first line Nigeria Africa',
    'HIV viral suppression PEPFAR Nigeria outcomes',
]

def fetch_pubmed(query, max_results=80):
    print('Fetching:', query[:55] + '...')
    try:
        handle = Entrez.esearch(db='pubmed', term=query, retmax=max_results, sort='relevance')
        record = Entrez.read(handle)
        handle.close()
        ids = record['IdList']
        if not ids:
            return []
        handle = Entrez.efetch(db='pubmed', id=ids, rettype='abstract', retmode='xml')
        records = Entrez.read(handle)
        handle.close()
        papers = []
        for article in records.get('PubmedArticle', []):
            try:
                medline = article['MedlineCitation']
                art = medline['Article']
                abstract = ' '.join(str(x) for x in art.get('Abstract', {}).get('AbstractText', []))
                if len(abstract) < 100:
                    continue
                pmid = str(medline['PMID'])
                title = str(art.get('ArticleTitle', ''))
                pub_date = art.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {})
                year = str(pub_date.get('Year', pub_date.get('MedlineDate', 'Unknown')))[:4]
                papers.append({
                    'pmid': pmid, 'title': title, 'abstract': abstract, 'year': year,
                    'source': 'PubMed PMID:' + pmid + ' (' + year + ')',
                    'url': 'https://pubmed.ncbi.nlm.nih.gov/' + pmid + '/'
                })
            except Exception:
                continue
        time.sleep(0.4)
        return papers
    except Exception as e:
        print('  Error:', e)
        return []

all_papers = []
for q in PUBMED_QUERIES:
    papers = fetch_pubmed(q)
    all_papers.extend(papers)
    print('  ->', len(papers), 'abstracts')

seen, unique = set(), []
for p in all_papers:
    if p['pmid'] not in seen:
        seen.add(p['pmid'])
        unique.append(p)

pubmed_path = os.path.join(DATA_DIR, 'pubmed_papers.json')
with open(pubmed_path, 'w') as f:
    json.dump(unique, f, indent=2)

print('\n✅ Saved', len(unique), 'unique papers to Drive')

Fetching: HIV antiretroviral therapy drug interactions clinical t...
  -> 72 abstracts
Fetching: PrEP HIV pre-exposure prophylaxis efficacy adherence...
  -> 80 abstracts
Fetching: HIV tuberculosis coinfection treatment outcomes...
  -> 79 abstracts
Fetching: long acting injectable cabotegravir rilpivirine ART...
  -> 80 abstracts
Fetching: HIV treatment adherence intervention sub-Saharan Africa...
  -> 79 abstracts
Fetching: TDF DTG dolutegravir lamivudine first line Nigeria Afri...
  -> 2 abstracts
Fetching: HIV viral suppression PEPFAR Nigeria outcomes...
  -> 19 abstracts

✅ Saved 408 unique papers to Drive


In [ ]:
import json, os


GUIDELINES = [
    {
        'guideline_label': 'WHO Global HIV Treatment Guidelines',
        'section': 'WHO Recommended First-Line ART Regimens',
        'content': 'WHO 2021 Consolidated Guidelines on HIV — Recommended First-Line ART:\n\nPreferred first-line regimen for adults and adolescents:\n- Tenofovir disoproxil fumarate 300mg + Lamivudine 300mg + Dolutegravir 50mg (TDF/3TC/DTG) — ONE tablet once daily\n- This is the preferred regimen globally including all low and middle income countries\n- Replaces the previous standard of TDF/3TC/EFV as first-line\n\nWhy TDF/3TC/DTG is WHO preferred:\n- Dolutegravir has a higher barrier to resistance than efavirenz\n- Superior viral suppression rates compared to efavirenz-based regimens\n- Better tolerability with fewer CNS side effects than efavirenz\n- Once daily single pill improves adherence\n- Fixed-dose combination available as a single tablet\n- Lower cost than TAF-based regimens used in high-income countries\n\nAlternative first-line regimens:\n- TDF/3TC/EFV 400mg — low-dose efavirenz, acceptable alternative\n- TDF/3TC/EFV 600mg — standard dose efavirenz still widely used\n- ABC/3TC/DTG — for patients with renal impairment where TDF is contraindicated',
        'source': 'WHO: Consolidated Guidelines on HIV 2021',
        'url': 'https://www.who.int/publications/i/item/9789240031593'
    },
    {
        'guideline_label': 'Nigeria HIV Treatment Guidelines',
        'section': 'Nigeria National ART Guidelines First-Line Regimens',
        'content': 'Nigeria National HIV/AIDS Treatment Guidelines — First-Line ART:\n\nRecommended first-line regimen in Nigeria:\n- TDF + 3TC + DTG (Tenofovir/Lamivudine/Dolutegravir) — preferred first-line for all treatment-naive adults\n- Available as fixed-dose combination — one tablet once daily\n- Dispensed through PEPFAR-supported ART sites including APIN, IHVN, CCCRN\n\nPrevious standard now used as alternative:\n- TDF/3TC/EFV 600mg — was first-line before DTG scale-up, still used in some settings\n\nSpecial populations:\n- Pregnant women: TDF/3TC/DTG recommended throughout pregnancy including first trimester\n- TB/HIV coinfection: TDF/3TC/EFV preferred when on rifampicin. DTG dose doubled to 50mg twice daily if DTG used with rifampicin\n- Renal impairment eGFR below 50: switch TDF to ABC — use ABC/3TC/DTG\n\nSecond-line ART:\n- After TDF/3TC/DTG failure: AZT/3TC + boosted PI (lopinavir/ritonavir or atazanavir/ritonavir)\n\nContext:\n- Nigeria has approximately 1.9 million PLHIV\n- PEPFAR supports approximately 90 percent of patients on ART in Nigeria\n- DTG-based regimens scaled up nationally from 2020',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
    {
        'guideline_label': 'Adult & Adolescent ARV Treatment',
        'section': 'First-Line ART Regimens Detailed',
        'content': 'DHHS Recommended First-Line ART Regimens for Treatment-Naive Adults:\n\nPreferred Regimens:\n1. Bictegravir 50mg + Tenofovir alafenamide 25mg + Emtricitabine 200mg (BIC/TAF/FTC) — Biktarvy. One pill once daily.\n2. Dolutegravir 50mg + Tenofovir alafenamide 25mg + Emtricitabine 200mg (DTG + TAF/FTC). Once daily.\n3. Dolutegravir 50mg + Lamivudine 300mg (DTG/3TC) — Dovato. Two-drug regimen only for patients with HIV RNA below 500,000 copies per mL and no HBV coinfection.\n\nAlternative Regimens:\n1. Darunavir/Cobicistat/TAF/FTC — when INSTI resistance is suspected\n2. Raltegravir 400mg twice daily + TDF/FTC\n3. Efavirenz 600mg + TDF/FTC — useful in TB coinfection on rifampicin\n\nNote: DHHS prefers TAF over TDF due to better renal and bone profile. WHO prefers TDF globally due to lower cost. Both are clinically effective.',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'Adult & Adolescent ARV Treatment',
        'section': 'When to Start ART',
        'content': 'ART is recommended for all individuals with HIV regardless of CD4 count. Early initiation reduces AIDS-defining events, non-AIDS morbidity, and HIV transmission.\n\nRapid ART initiation on the same day or within days of diagnosis is recommended for most patients. Evidence from HPTN 052 and START trials confirmed that early ART significantly reduces both AIDS and serious non-AIDS events.\n\nExceptions where initiation may be deferred:\n- Active cryptococcal meningitis: defer 4 to 6 weeks to avoid IRIS\n- Active TB meningitis: defer 8 weeks\n- Need for patient education and treatment readiness assessment',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'Adult & Adolescent ARV Treatment',
        'section': 'Drug Interactions Rifampicin and ARVs',
        'content': 'Rifampicin is a potent inducer of CYP3A4 and P-glycoprotein and significantly reduces plasma concentrations of many ARVs.\n\nKey interactions:\n- Rifampicin reduces dolutegravir AUC by approximately 54 percent. Dolutegravir dose should be increased to 50mg twice daily.\n- Rifampicin reduces efavirenz levels modestly. Standard 600mg dose is generally maintained.\n- Rifampicin is contraindicated with most protease inhibitors and cobicistat-boosted regimens.\n- Rifabutin is preferred over rifampicin when a PI-based regimen is required.\n\nFor HIV/TB coinfected patients on rifampicin:\n- Preferred: efavirenz-based regimen TDF/3TC/EFV\n- Alternative: dolutegravir 50mg twice daily plus TDF/3TC',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'Adult & Adolescent ARV Treatment',
        'section': 'Switching and Simplifying ART Regimens',
        'content': 'Virologically suppressed patients may switch ART for toxicity, drug interactions, pill burden, or cost.\n\nKey considerations:\n- Confirm viral suppression before switching\n- Review resistance history as prior resistance may limit options\n- TAF-based regimens preferred over TDF in renal impairment or osteoporosis per DHHS\n- TDF remains preferred in WHO and Nigerian guidelines due to cost\n- DTG/3TC is effective 2-drug maintenance for stable patients without HBV or prior NRTI resistance\n\nTANGO trial: DTG/3TC was non-inferior to TAF-based 3-drug regimens through 144 weeks.',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'Adult & Adolescent ARV Treatment',
        'section': 'HIV and Renal Disease',
        'content': 'HIV-associated nephropathy and TDF-related nephrotoxicity are key renal considerations.\n\nTDF can cause proximal tubular dysfunction and reduced GFR. TAF achieves higher intracellular concentrations at lower plasma levels with significantly less renal and bone toxicity.\n\nRecommendations:\n- DHHS: Use TAF-based regimens if eGFR below 60 mL per min\n- WHO/Nigeria: Switch from TDF to ABC if eGFR below 50 mL per min\n- Monitor renal function every 6 months on TDF\n- Avoid TDF if eGFR below 30 mL per min\n- Screen for HIVAN with urine protein to creatinine ratio at baseline and annually',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'Adult & Adolescent ARV Treatment',
        'section': 'HIV Drug Resistance Testing',
        'content': 'Resistance testing is recommended at HIV diagnosis and before initiating or changing ART.\n\nTypes:\n- Genotypic testing sequences RT, protease, and integrase genes. Faster and cheaper.\n- Phenotypic testing directly measures viral replication in drug. Used for complex resistance.\n\nIndications:\n- All patients at entry into care\n- Before ART initiation in treatment-naive patients\n- Virologic failure defined as HIV RNA above 200 copies per mL on ART\n\nKey mutations: M184V confers 3TC/FTC resistance, K65R confers TDF resistance, Q148H/R/K confers INSTI resistance, K103N confers NNRTI resistance.',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'Long-Acting Injectable ART',
        'section': 'Cabotegravir and Rilpivirine Long-Acting Injectable',
        'content': 'Long-acting injectable cabotegravir plus rilpivirine known as CAB+RPV LA or Cabenuva is approved for virologically stable adults.\n\nATLAS and FLAIR trial data:\n- CAB+RPV LA every 2 months was non-inferior to daily oral ART\n- 94 percent of participants preferred injectable over daily oral therapy\n- Injection site reaction discontinuation rate was approximately 2 percent\n\nEligibility:\n- Virologically suppressed with HIV RNA below 50 copies per mL\n- No history of treatment failure\n- No resistance to cabotegravir or rilpivirine\n- No HBV coinfection\n\nATLAS-2M confirmed every-2-month dosing is non-inferior to monthly dosing.\n\nNote: Currently limited availability in Nigeria and most African settings due to cost.',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'PrEP Guidelines',
        'section': 'PrEP Efficacy and Indications',
        'content': 'Pre-exposure prophylaxis with TDF/FTC or TAF/FTC is highly effective in preventing HIV acquisition.\n\nKey trials:\n- iPrEx: TDF/FTC reduced HIV incidence by 44 percent overall and more than 90 percent with detectable drug levels\n- Partners PrEP: 75 percent efficacy in serodiscordant heterosexual couples\n- DISCOVER: TAF/FTC non-inferior to TDF/FTC in MSM and transgender women\n- PURPOSE 1 and PURPOSE 2: near-complete protection in cisgender women\n\nIndications include adults and adolescents at substantial HIV risk, serodiscordant couples, MSM with inconsistent condom use or recent STI, and people who inject drugs.\n\nAdherence is the strongest predictor of PrEP efficacy.',
        'source': 'CDC: PrEP Clinical Practice Guidelines',
        'url': 'https://www.cdc.gov/hiv/clinicians/prevention/prep.html'
    },
    {
        'guideline_label': 'PrEP Guidelines',
        'section': 'PrEP in Special Populations',
        'content': 'PrEP uptake varies significantly across populations due to structural and social barriers.\n\nBlack and Latino MSM:\n- High HIV incidence but lower PrEP uptake than white MSM\n- Barriers include medical mistrust, insurance gaps, stigma, and provider bias\n\nTransgender women:\n- TAF/FTC preferred due to absence of interaction with feminizing hormones\n\nPeople who inject drugs:\n- TDF/FTC reduces HIV incidence\n- Integrated harm reduction and PrEP programs improve retention\n\nAdolescents:\n- FDA-approved for adolescents weighing at least 35kg\n- School-based and youth-friendly clinics improve adherence\n\nNigeria and sub-Saharan Africa:\n- PrEP scale-up ongoing through PEPFAR supported programs\n- TDF/FTC used as PrEP in Nigeria — TAF not widely available',
        'source': 'CDC: PrEP Clinical Practice Guidelines',
        'url': 'https://www.cdc.gov/hiv/clinicians/prevention/prep.html'
    },
    {
        'guideline_label': 'HIV in Pregnancy',
        'section': 'ART in Pregnancy Recommendations',
        'content': 'All pregnant people with HIV should receive ART regardless of CD4 count or viral load for health and PMTCT.\n\nPreferred regimens in pregnancy:\n- WHO and Nigeria FMOH: TDF/3TC/DTG preferred throughout pregnancy including first trimester\n- DHHS: DTG-based regimens preferred. Neural tube defect risk low at approximately 0.1 percent.\n- Raltegravir plus TDF/FTC remains an alternative\n- Efavirenz-based regimens acceptable if started before conception\n\nGoals:\n- Achieve viral suppression below 200 copies per mL before delivery\n- Undetectable viral load reduces perinatal transmission to below 1 percent\n- Cesarean delivery recommended if viral load exceeds 1000 copies per mL near delivery',
        'source': 'AIDSinfo: DHHS Perinatal HIV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/perinatal'
    },
    {
        'guideline_label': 'Opportunistic Infection Prevention',
        'section': 'PCP and Toxoplasmosis Prophylaxis',
        'content': 'PCP prophylaxis is indicated for CD4 below 200 cells per mm3, CD4 percentage below 14 percent, or prior PCP episode.\n\nPreferred: TMP-SMX one double-strength tablet daily.\nAlternatives for sulfa-allergic patients: Dapsone, atovaquone, or aerosolized pentamidine.\n\nToxoplasma prophylaxis:\n- Indicated for Toxoplasma IgG-positive patients with CD4 below 100 cells per mm3\n- TMP-SMX DS daily also covers Toxoplasma\n- Discontinue when CD4 exceeds 200 cells per mm3 for more than 3 months on ART',
        'source': 'AIDSinfo: Guidelines for Prevention of Opportunistic Infections',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/hiv-clinical-guidelines-adult-adolescent-opportunistic-infection'
    },
    {
        'guideline_label': 'Opportunistic Infection Prevention',
        'section': 'Cryptococcal Meningitis Management',
        'content': 'Cryptococcus neoformans meningitis is a leading cause of mortality in HIV with CD4 below 100 cells per mm3.\n\nPrevention:\n- CrAg screening for CD4 below 100 cells per mm3 in high-prevalence settings including Nigeria\n- Fluconazole pre-emptive therapy for CrAg-positive patients\n\nTreatment:\n- Induction: Liposomal amphotericin B plus flucytosine for 2 weeks\n- Consolidation: Fluconazole 400mg daily for 8 weeks\n- Maintenance: Fluconazole 200mg daily until CD4 exceeds 200 for more than 1 year on ART\n\nDefer ART 4 to 6 weeks after antifungal initiation to reduce IRIS risk.',
        'source': 'AIDSinfo: Guidelines for Prevention of Opportunistic Infections',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/hiv-clinical-guidelines-adult-adolescent-opportunistic-infection'
    },
    {
        'guideline_label': 'Opportunistic Infection Prevention',
        'section': 'CMV and MAC Prophylaxis',
        'content': 'MAC prophylaxis:\n- Indicated for CD4 below 50 cells per mm3\n- Preferred: Azithromycin 1200mg weekly\n- Discontinue when CD4 exceeds 100 cells per mm3 for more than 3 months on ART\n\nCMV:\n- Routine prophylaxis not recommended\n- CMV retinitis screening for CD4 below 50 cells per mm3\n- Treatment: IV ganciclovir or oral valganciclovir\n- Secondary prophylaxis until CD4 exceeds 100 to 150 cells per mm3',
        'source': 'AIDSinfo: Guidelines for Prevention of Opportunistic Infections',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/hiv-clinical-guidelines-adult-adolescent-opportunistic-infection'
    },
    {
        'guideline_label': 'HIV TB Coinfection',
        'section': 'TB Treatment in HIV-Positive Patients',
        'content': 'TB is the leading cause of death among HIV-positive individuals globally and particularly in Nigeria.\n\nART timing:\n- CD4 below 50 cells per mm3: start ART within 2 weeks of TB treatment\n- CD4 at or above 50: start ART within 8 weeks\n- TB meningitis: defer ART 8 weeks regardless of CD4\n\nPreferred ART with rifampicin in Nigeria:\n- TDF/3TC/EFV 600mg — efavirenz not significantly affected by rifampicin\n- Alternative: TDF/3TC + DTG 50mg twice daily — dose doubled due to rifampicin induction\n\nIRIS management: NSAIDs or corticosteroids for severe cases. Continue both ART and TB treatment.',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines TB Coinfection',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
    {
        'guideline_label': 'HIV and Mental Health',
        'section': 'Depression Adherence and Mental Health in HIV',
        'content': 'Mental health comorbidities are highly prevalent in PLHIV and directly impact ART adherence.\n\nPrevalence:\n- Depression affects 30 to 40 percent of PLHIV compared to approximately 7 percent in general population\n- Anxiety, PTSD, and substance use disorders are also elevated\n\nImpact:\n- Depression is one of the strongest predictors of ART non-adherence\n- Untreated depression linked to faster disease progression and higher mortality\n\nManagement:\n- Screen with PHQ-9 at entry to care and annually\n- SSRIs are first-line. Check ARV drug interactions.\n- Efavirenz associated with neuropsychiatric side effects. Consider switching.\n- Integrated HIV and mental health care improves adherence and psychiatric outcomes',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
    {
        'guideline_label': 'HIV Prevention',
        'section': 'Treatment as Prevention and Undetectable Equals Untransmittable',
        'content': 'Treatment as prevention is a cornerstone of HIV prevention globally.\n\nEvidence:\n- PARTNER, PARTNER2, and Opposites Attract studies observed zero linked transmissions from virologically suppressed individuals\n- Applies to heterosexual and MSM serodiscordant couples\n- Requires sustained viral suppression below 200 copies per mL\n\nHPTN 052: early ART reduced transmission to partners by 96 percent.\n\nPublic health implications:\n- UNAIDS 95-95-95 targets: 95 percent diagnosed, 95 percent on ART, 95 percent virally suppressed\n- Nigeria 95-95-95 progress: significant gaps remain particularly in the second and third 95\n- Test and treat combined with PrEP provides the most effective combination prevention',
        'source': 'AIDSinfo: DHHS Adult & Adolescent ARV Guidelines',
        'url': 'https://clinicalinfo.hiv.gov/en/guidelines/adult-adolescent-arv'
    },
]

guideline_path = os.path.join(DATA_DIR, 'hiv_guidelines.json')
with open(guideline_path, 'w') as f:
    json.dump(GUIDELINES, f, indent=2)

print('✅ Saved', len(GUIDELINES), 'guideline sections')
for g in GUIDELINES:
    print('  *', g['section'])

✅ Saved 18 guideline sections
  * WHO Recommended First-Line ART Regimens
  * Nigeria National ART Guidelines First-Line Regimens
  * First-Line ART Regimens Detailed
  * When to Start ART
  * Drug Interactions Rifampicin and ARVs
  * Switching and Simplifying ART Regimens
  * HIV and Renal Disease
  * HIV Drug Resistance Testing
  * Cabotegravir and Rilpivirine Long-Acting Injectable
  * PrEP Efficacy and Indications
  * PrEP in Special Populations
  * ART in Pregnancy Recommendations
  * PCP and Toxoplasmosis Prophylaxis
  * Cryptococcal Meningitis Management
  * CMV and MAC Prophylaxis
  * TB Treatment in HIV-Positive Patients
  * Depression Adherence and Mental Health in HIV
  * Treatment as Prevention and Undetectable Equals Untransmittable


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Initialize your HuggingFace embeddings model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Gather your documents (your loaded guidelines + pubmed papers)
docs = []
for g in GUIDELINES:
    docs.append(Document(page_content=g['content'], metadata={'source': g['source']}))

# 3. Build and persist the Chroma database to your target directory
db_path = '/content/drive/MyDrive/Colab Notebooks/RAG-LLM project/hiv_chroma_db'
os.makedirs(db_path, exist_ok=True)

vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=db_path
)

print(f"✅ Chroma database successfully created and saved at: {db_path}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Chroma database successfully created and saved at: /content/drive/MyDrive/Colab Notebooks/RAG-LLM project/hiv_chroma_db


In [ ]:
import json, os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

with open(os.path.join(DATA_DIR, 'pubmed_papers.json')) as f:
    papers = json.load(f)
with open(os.path.join(DATA_DIR, 'hiv_guidelines.json')) as f:
    guidelines = json.load(f)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700, chunk_overlap=100,
    separators=['\n\n', '\n', '. ', ' ', '']
)

documents = []

for p in papers:
    text = 'Title: ' + p['title'] + '\n\nAbstract: ' + p['abstract']
    for chunk in splitter.split_text(text):
        documents.append(Document(
            page_content=chunk,
            metadata={
                'source': p['source'], 'url': p['url'],
                'year': p['year'], 'type': 'pubmed'
            }
        ))

for g in guidelines:
    text = 'Guideline: ' + g['guideline_label'] + '\nSection: ' + g['section'] + '\n\n' + g['content']
    for chunk in splitter.split_text(text):
        documents.append(Document(
            page_content=chunk,
            metadata={
                'source': g['source'], 'url': g['url'],
                'type': 'guideline', 'section': g['section']
            }
        ))

print('✅ Chunking complete')
print('Total documents:', len(documents))
print('  PubMed:   ', sum(1 for d in documents if d.metadata['type'] == 'pubmed'))
print('  Guideline:', sum(1 for d in documents if d.metadata['type'] == 'guideline'))

✅ Chunking complete
Total documents: 1812
  PubMed:    1784
  Guideline: 28


In [ ]:
import json, os

BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/HIV-RAG-LLM'
DATA_DIR = os.path.join(BASE_DIR, 'data')

with open(os.path.join(DATA_DIR, 'hiv_guidelines.json')) as f:
    existing = json.load(f)

EXTRA_NIGERIA = [
    {
        'guideline_label': 'Nigeria HIV Treatment Guidelines',
        'section': 'Nigeria HIV Program Structure and Implementing Partners',
        'content': 'Nigeria HIV program is one of the largest in the world supported primarily through PEPFAR funding.\n\nKey implementing partners in Nigeria:\n- APIN Public Health Initiatives — supports ART sites across multiple states\n- Institute of Human Virology Nigeria (IHVN) — supports sites in FCT, Niger, Kogi and others\n- Centre for Integrated Health Programs (CIHP) — supports northern Nigeria states\n- Catholic Caritas Foundation of Nigeria (CCCRN) — supports southeastern states\n- Achieving Health Nigeria Initiative (AHNi) — PEPFAR implementing partner\n- Family Health International (FHI 360) — supports community programs\n\nProgram targets:\n- Nigeria PEPFAR targets approximately 2 million patients on ART\n- UNAIDS estimates 1.9 million PLHIV in Nigeria as of 2022\n- Treatment coverage gap remains significant — estimated 300,000 to 400,000 PLHIV not yet on ART\n- Key gaps in the cascade: linkage to care after HIV testing and retention in care\n\nFunding:\n- PEPFAR provides approximately 90 percent of HIV treatment funding in Nigeria\n- Global Fund supports community-based programs and TB/HIV integration\n- Federal government of Nigeria contributes through NACA and state governments',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
    {
        'guideline_label': 'Nigeria HIV Treatment Guidelines',
        'section': 'Nigeria Viral Load Monitoring and Treatment Failure',
        'content': 'Viral load monitoring in Nigeria:\n\nWHO and Nigeria FMOH recommendations for viral load monitoring:\n- Baseline viral load at ART initiation\n- Viral load at 6 months after ART initiation\n- Viral load at 12 months\n- Then annually for stable patients\n- Repeat viral load 3 months after enhanced adherence counseling if first viral load above 1000 copies per mL\n\nVirologic failure definition in Nigeria:\n- Confirmed HIV RNA above 1000 copies per mL after at least 6 months on ART\n- Must rule out poor adherence before switching regimens\n\nEnhanced adherence counseling (EAC):\n- Three sessions of EAC over 3 months before switching to second-line\n- EAC sessions should address barriers to adherence including transportation, stigma, food insecurity, and side effects\n- Only switch to second-line after confirmed virologic failure following EAC\n\nViral load infrastructure in Nigeria:\n- Point-of-care viral load testing being scaled up through PEPFAR\n- Many sites still rely on central laboratory testing with 2 to 4 week turnaround\n- Dried blood spot (DBS) sampling used in remote sites without venipuncture capacity\n\nTreatment failure in Nigeria:\n- Most common causes: poor adherence, drug stock-outs, regimen toxicity\n- NRTI resistance mutations particularly M184V common after first-line failure\n- INSTI resistance rare but emerging — document and report',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
    {
        'guideline_label': 'Nigeria HIV Treatment Guidelines',
        'section': 'Nigeria PMTCT Program and Elimination of Mother to Child Transmission',
        'content': 'Prevention of Mother to Child Transmission (PMTCT) in Nigeria:\n\nNigeria has one of the highest burdens of mother-to-child HIV transmission globally.\n\nNigeria PMTCT cascade challenges:\n- Antenatal care attendance is improving but gaps remain in rural areas\n- HIV testing uptake in ANC is above 90 percent in PEPFAR-supported sites\n- ART initiation among HIV-positive pregnant women has improved significantly since DTG scale-up\n- Postpartum retention remains the weakest point — many mothers lost to follow-up after delivery\n\nOption B+ in Nigeria:\n- Nigeria adopted Option B+ in 2013 — all HIV-positive pregnant women start lifelong ART regardless of CD4 count\n- Current preferred regimen: TDF/3TC/DTG throughout pregnancy including first trimester\n- Previous concern about neural tube defects with periconceptional DTG was not confirmed in subsequent larger studies\n\nInfant prophylaxis in Nigeria:\n- All HIV-exposed infants receive nevirapine from birth\n- High risk infants receive dual prophylaxis — nevirapine plus AZT\n- Early infant diagnosis (EID) at 6 weeks using PCR-based testing\n- HIV-exposed infants tested at 6 weeks, 9 months, and 18 months\n\nBreastfeeding in Nigeria:\n- WHO and Nigeria FMOH recommend breastfeeding with maternal ART for HIV-positive mothers\n- Maternal viral suppression makes breastfeeding transmission risk very low below 1 percent\n- Exclusive breastfeeding for 6 months then complementary feeding while continuing to breastfeed until 12 months\n- Maternal ART must be continued throughout breastfeeding period',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines PMTCT',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
    {
        'guideline_label': 'Nigeria HIV Treatment Guidelines',
        'section': 'Nigeria HIV Key Populations Program',
        'content': 'Key populations HIV programming in Nigeria:\n\nKey populations with highest HIV burden in Nigeria:\n- Female sex workers (FSW): HIV prevalence 14 to 20 percent in surveys\n- Men who have sex with men (MSM): HIV prevalence 23 to 25 percent — highly criminalized in Nigeria\n- People who inject drugs (PWID): HIV prevalence 3 to 5 percent\n- Transgender persons: very limited data available\n\nProgrammatic challenges:\n- Same-sex relationships are criminalized under the Same Sex Marriage Prohibition Act 2014 — creates significant barriers to MSM accessing HIV services\n- Stigma and discrimination prevent key populations from accessing mainstream health facilities\n- Key population-led and community-based services are essential for reaching these groups\n\nPrEP for key populations in Nigeria:\n- TDF/FTC PrEP available at select sites through PEPFAR-supported key population programs\n- Demand creation and linkage to care remains a challenge\n- FSW and MSM have highest benefit from PrEP due to elevated HIV incidence\n\nHarm reduction:\n- Needle and syringe programs available in limited locations\n- Opioid agonist therapy not widely available in Nigeria\n- Community outreach and peer-led services most effective for PWID in Nigerian context',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines Key Populations',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
    {
        'guideline_label': 'Nigeria HIV Treatment Guidelines',
        'section': 'Nigeria ART Drug Supply and Commodity Security',
        'content': 'HIV drug supply and commodity security in Nigeria:\n\nDrug procurement in Nigeria:\n- Most ARVs procured through PEPFAR and Global Fund mechanisms\n- NAFDAC (National Agency for Food and Drug Administration and Control) regulates ARV quality\n- Generic fixed-dose combinations (FDCs) procured through PEPFAR GHSC-PSM supply chain\n- TDF/3TC/DTG FDC (also known as Telura or equivalent generics) is the main first-line drug procured\n\nDrug stock-out challenges:\n- Stock-outs are a significant cause of treatment interruption and virologic failure in Nigeria\n- Decentralized drug management at facility level reduces stock-out risk\n- Patients encouraged to maintain at least 2-week buffer supply\n- Multi-month dispensing (MMD) — 3 to 6 months supply — reduces stock-out risk and clinic visits\n\nMulti-month dispensing (MMD) in Nigeria:\n- PEPFAR-supported sites have scaled up 3-month and 6-month dispensing for stable patients\n- MMD improves retention in care and reduces transportation burden for patients\n- Stable patient defined as: on ART more than 6 months, virologically suppressed, no opportunistic infections\n- Community ART delivery and differentiated service delivery models expanding in Nigeria\n\nSecond-line and third-line drug availability:\n- Lopinavir/ritonavir (LPV/r) available for second-line through PEPFAR procurement\n- Atazanavir/ritonavir increasingly available as preferred boosted PI\n- Third-line drugs such as darunavir/ritonavir have very limited availability in Nigeria\n- Dolutegravir-based second-line regimens being evaluated but not yet standard in Nigeria',
        'source': 'Nigeria FMOH: National HIV/AIDS Treatment Guidelines Drug Supply',
        'url': 'https://naca.gov.ng/treatment-guidelines/'
    },
]

all_guidelines = existing + EXTRA_NIGERIA
with open(os.path.join(DATA_DIR, 'hiv_guidelines.json'), 'w') as f:
    json.dump(all_guidelines, f, indent=2)

print('Previous sections:', len(existing))
print('New Nigeria sections added:', len(EXTRA_NIGERIA))
print('Total sections:', len(all_guidelines))
print('\nNew sections:')
for s in EXTRA_NIGERIA:
    print('  *', s['section'])

Previous sections: 18
New Nigeria sections added: 5
Total sections: 23

New sections:
  * Nigeria HIV Program Structure and Implementing Partners
  * Nigeria Viral Load Monitoring and Treatment Failure
  * Nigeria PMTCT Program and Elimination of Mother to Child Transmission
  * Nigeria HIV Key Populations Program
  * Nigeria ART Drug Supply and Commodity Security


In [ ]:
from IPython.display import display, HTML

def display_result(question, result):
    conflict_color = '#ff6b6b' if result['has_conflict'] else '#00d4aa'
    conflict_text = '⚠️ CONFLICT DETECTED' if result['has_conflict'] else '✅ NO CONFLICT'
    conflict_msg = 'Guidelines and research present differing positions. Review carefully before clinical decision making.' if result['has_conflict'] else 'Guidelines and research are in agreement on this topic.'

    guidelines = [s for s in result['sources'] if s['type'] == 'guideline']
    pubmed     = [s for s in result['sources'] if s['type'] == 'pubmed']

    guideline_refs = ''
    for s in guidelines:
        guideline_refs += f'''
        <div style="margin-bottom:10px; padding:10px; border-left:3px solid #00d4aa; background:#0d2137;">
            <div style="color:#00d4aa; font-weight:600; font-size:12px;">[{s["citation"]}]</div>
            <div style="color:#94a3b8; font-size:11px; margin-top:4px;">{s.get("section","")}</div>
            <a href="{s["url"]}" target="_blank" style="color:#64748b; font-size:10px; font-family:monospace;">{s["url"]}</a>
        </div>'''

    pubmed_refs = ''
    for s in pubmed:
        pubmed_refs += f'''
        <div style="margin-bottom:10px; padding:10px; border-left:3px solid #0ea5e9; background:#0d2137;">
            <div style="color:#0ea5e9; font-weight:600; font-size:12px;">[{s["citation"]}]</div>
            <div style="color:#94a3b8; font-size:11px; margin-top:4px;">{s["source"]}</div>
            <a href="{s["url"]}" target="_blank" style="color:#64748b; font-size:10px; font-family:monospace;">{s["url"]}</a>
        </div>'''

    # Format answer sections
    answer = result['answer']
    for section in ['GUIDELINE POSITION:', 'RESEARCH EVIDENCE:', 'CONFLICT OR AGREEMENT:', 'CLINICAL IMPLICATION:']:
        answer = answer.replace(section, f'<div style="color:#00d4aa; font-weight:700; font-size:13px; margin-top:18px; margin-bottom:6px; letter-spacing:0.05em;">{section}</div>')

    html = f'''
    <div style="font-family: Inter, sans-serif; background:#0a0e17; color:#e2e8f0; padding:24px; border-radius:12px; max-width:900px; margin:10px 0;">

        <!-- Header -->
        <div style="border-bottom:1px solid #1e2d40; padding-bottom:14px; margin-bottom:20px;">
            <div style="font-size:11px; color:#00d4aa; font-family:monospace; letter-spacing:0.1em; text-transform:uppercase; margin-bottom:6px;">HIV Clinical Evidence Assistant</div>
            <div style="font-size:15px; font-weight:600; color:#f1f5f9;">QUESTION</div>
            <div style="font-size:14px; color:#cbd5e1; margin-top:4px;">{question}</div>
        </div>

        <!-- Conflict Banner -->
        <div style="padding:12px 16px; border-radius:8px; background:{conflict_color}20; border:1px solid {conflict_color}; margin-bottom:20px;">
            <div style="font-weight:700; color:{conflict_color}; font-size:13px;">{conflict_text}</div>
            <div style="color:#94a3b8; font-size:12px; margin-top:4px;">{conflict_msg}</div>
        </div>

        <!-- Answer -->
        <div style="background:#111827; border:1px solid #1e2d40; border-radius:8px; padding:20px; margin-bottom:20px; line-height:1.8; font-size:13.5px;">
            {answer}
        </div>

        <!-- References -->
        <div style="background:#111827; border:1px solid #1e2d40; border-radius:8px; padding:20px;">
            <div style="font-size:11px; color:#64748b; font-family:monospace; letter-spacing:0.1em; text-transform:uppercase; margin-bottom:14px;">References</div>

            <div style="font-size:12px; font-weight:600; color:#00d4aa; margin-bottom:8px;">Clinical Practice Guidelines</div>
            {guideline_refs if guideline_refs else '<div style="color:#64748b; font-size:12px;">None retrieved</div>'}

            <div style="font-size:12px; font-weight:600; color:#0ea5e9; margin-top:16px; margin-bottom:8px;">Primary Literature (PubMed)</div>
            {pubmed_refs if pubmed_refs else '<div style="color:#64748b; font-size:12px;">None retrieved</div>'}

            <div style="margin-top:16px; padding-top:12px; border-top:1px solid #1e2d40; font-size:10px; color:#475569;">
                ⚠️ For clinical evidence research and education only. Not a substitute for professional clinical judgment or official guidelines.
            </div>
        </div>
    </div>
    '''
    display(HTML(html))

print('✅ Display formatter ready')

✅ Display formatter ready


In [ ]:
import os, pickle, chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print('✅ Embedding model ready')

chroma_client = chromadb.EphemeralClient()

chunks    = [d.page_content for d in documents]
metadatas = [d.metadata for d in documents]

BATCH_SIZE = 300
vectorstore = None
total = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(chunks), BATCH_SIZE):
    bc = chunks[i:i+BATCH_SIZE]
    bm = metadatas[i:i+BATCH_SIZE]
    print('Batch', (i//BATCH_SIZE)+1, 'of', total)
    if vectorstore is None:
        vectorstore = Chroma.from_texts(
            texts=bc, embedding=embeddings, metadatas=bm,
            client=chroma_client, collection_name='hiv_evidence'
        )
    else:
        vectorstore.add_texts(texts=bc, metadatas=bm)

retriever = vectorstore.as_retriever(search_kwargs={'k': 8})

print('✅ Vectorstore built —', vectorstore._collection.count(), 'documents indexed')
print('✅ Retriever ready')

with open(os.path.join(DATA_DIR, 'vectorstore.pkl'), 'wb') as f:
    pickle.dump({'chunks': chunks, 'metadatas': metadatas}, f)
print('✅ Chunks saved to Drive for fast reload on restart')

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model ready
Batch 1 of 7
Batch 2 of 7
Batch 3 of 7
Batch 4 of 7
Batch 5 of 7
Batch 6 of 7
Batch 7 of 7
✅ Vectorstore built — 1812 documents indexed
✅ Retriever ready
✅ Chunks saved to Drive for fast reload on restart


In [ ]:
import os, pickle, chromadb
from google.colab import drive
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/HIV-RAG-LLM'
DATA_DIR = os.path.join(BASE_DIR, 'data')

print('Loading chunks from Drive...')
with open(os.path.join(DATA_DIR, 'vectorstore.pkl'), 'rb') as f:
    saved = pickle.load(f)

chunks    = saved['chunks']
metadatas = saved['metadatas']
print('Chunks loaded:', len(chunks))

print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

chroma_client = chromadb.EphemeralClient()
BATCH_SIZE = 300
vectorstore = None
total = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(0, len(chunks), BATCH_SIZE):
    bc = chunks[i:i+BATCH_SIZE]
    bm = metadatas[i:i+BATCH_SIZE]
    print('Batch', (i//BATCH_SIZE)+1, 'of', total)
    if vectorstore is None:
        vectorstore = Chroma.from_texts(
            texts=bc, embedding=embeddings, metadatas=bm,
            client=chroma_client, collection_name='hiv_evidence'
        )
    else:
        vectorstore.add_texts(texts=bc, metadatas=bm)

retriever = vectorstore.as_retriever(search_kwargs={'k': 8})
print('✅ Vectorstore ready —', vectorstore._collection.count(), 'documents')
print('✅ Retriever ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading chunks from Drive...
Chunks loaded: 1812
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batch 1 of 7
Batch 2 of 7
Batch 3 of 7
Batch 4 of 7
Batch 5 of 7
Batch 6 of 7
Batch 7 of 7
✅ Vectorstore ready — 5436 documents
✅ Retriever ready


In [ ]:
import os, re, time
from google.colab import userdata
from langchain_groq import ChatGroq

# 1. Load Groq API Key
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Key loaded from Colab Secrets")
except Exception:
    os.environ["GROQ_API_KEY"] = "paste-your-groq-key-here"
    print("✅ Key loaded from direct input")

# 2. Initialize Groq LLM
llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0.2)

# Connectivity test
test_response = llm.invoke("Say exactly: connected")
print("✅ Groq connected:", test_response.content.strip())

SYSTEM_PROMPT = """You are an HIV/AIDS clinical evidence research assistant supporting researchers, clinicians, pharmacists and MPH professionals globally with specific relevance to Nigeria and sub-Saharan Africa.

IMPORTANT CONTEXT:
- In Nigeria and most of sub-Saharan Africa the standard first-line ART regimen is TDF/3TC/DTG per WHO 2021 and Nigerian FMOH guidelines
- TAF-based regimens are preferred in US DHHS guidelines but TDF remains WHO preferred globally due to cost
- Always present BOTH WHO/Nigerian guidelines AND DHHS guidelines when relevant
- Explicitly state when recommendations differ between high-income and low-income country guidelines

Structure EVERY response using EXACTLY these four sections with these exact headings:

GUIDELINE POSITION:
Present WHO/Nigerian AND DHHS recommendations where they differ. Label which applies to which setting. Use citation tags like [WHO Guidelines 2021], [Nigeria FMOH Guidelines], [DHHS ARV Guidelines].

RESEARCH EVIDENCE:
What published research says. Reference studies by name and year where possible. Include study design, population, and key findings. Prioritise African studies where available.

CONFLICT OR AGREEMENT:
State explicitly whether research supports, contradicts, or adds nuance to the guideline. Note differences between WHO and DHHS recommendations and explain why they exist. This section must always appear even if there is no conflict — state agreement clearly.

CLINICAL IMPLICATION:
Present considerations for both resource-rich and resource-limited settings. State clearly that Nigerian clinicians should follow WHO and FMOH guidelines not DHHS. Do not make patient-specific decisions. End with a disclaimer that this is for research and education only.

Citation rules:
- Use descriptive citation tags like [WHO Guidelines 2021], [Nigeria FMOH Guidelines], [DHHS ARV Guidelines], [ADVANCE Trial], [NAMSAL Trial], [Partners PrEP Trial], [HPTN 052]
- Never use generic labels like Source 1 or Source 2
- Never invent citations or study names
- Only cite what is actually present in the retrieved evidence
- Do NOT include any thinking or reasoning process in your response
- Start your response immediately with GUIDELINE POSITION: with no preamble whatsoever"""


def parse_citation_tag(source_str, doc_type):
    if doc_type == "guideline":
        if "WHO" in source_str:
            return "WHO Guidelines 2021"
        elif "Nigeria" in source_str or "NACA" in source_str:
            return "Nigeria FMOH Guidelines"
        elif "Perinatal" in source_str or "Pregnant" in source_str:
            return "DHHS Perinatal Guidelines"
        elif "Opportunistic" in source_str or "OI" in source_str:
            return "DHHS OI Guidelines"
        elif "PrEP" in source_str and "CDC" in source_str:
            return "CDC PrEP Guidelines"
        elif "CDC" in source_str:
            return "CDC Guidelines"
        else:
            return "DHHS ARV Guidelines"
    else:
        year_match = re.search(r"\((\d{4})\)", source_str)
        year = year_match.group(1) if year_match else "n.d."
        return "PubMed " + year


def hiv_rag(question):
    all_docs = retriever.invoke(question)
    guideline_docs = [d for d in all_docs if d.metadata.get("type") == "guideline"][:3]
    pubmed_docs    = [d for d in all_docs if d.metadata.get("type") == "pubmed"][:3]

    if not guideline_docs and not pubmed_docs:
        return {
            "answer": "No relevant evidence retrieved for this question.",
            "sources": [],
            "has_conflict": False,
        }

    guideline_citations = [parse_citation_tag(d.metadata.get("source", ""), "guideline") for d in guideline_docs]
    pubmed_citations    = [parse_citation_tag(d.metadata.get("source", ""), "pubmed") for d in pubmed_docs]

    guideline_context = "GUIDELINE EVIDENCE:\n\n"
    if guideline_docs:
        for doc, tag in zip(guideline_docs, guideline_citations):
            guideline_context += "[" + tag + "]\nURL: " + doc.metadata.get("url", "") + "\n\n" + doc.page_content + "\n\n"
    else:
        guideline_context += "No guideline sections retrieved for this question.\n\n"

    pubmed_context = "RESEARCH EVIDENCE (PubMed):\n\n"
    if pubmed_docs:
        for doc, tag in zip(pubmed_docs, pubmed_citations):
            pubmed_context += "[" + tag + "]\nURL: " + doc.metadata.get("url", "") + "\n\n" + doc.page_content + "\n\n"
    else:
        pubmed_context += "No PubMed abstracts retrieved for this question.\n\n"

    full_prompt = (
        SYSTEM_PROMPT + "\n\n" +
        "Use the retrieved evidence below to answer the clinical question.\n\n" +
        guideline_context + pubmed_context +
        "\nQUESTION:\n" + question + "\n\n" +
        "Start immediately with GUIDELINE POSITION: — no thinking, no preamble, no reasoning steps. "
        "Use the exact citation tags shown in square brackets. "
        "Never use Source 1 or Source 2. "
        "Follow all four sections exactly: GUIDELINE POSITION, RESEARCH EVIDENCE, CONFLICT OR AGREEMENT, CLINICAL IMPLICATION."
    )

    # Retry loop
    max_retries = 3
    for attempt in range(max_retries):
        try:
            res = llm.invoke(full_prompt)
            answer = res.content

            # Strip thinking block if model includes it
            answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()

            # Cut anything before GUIDELINE POSITION
            if 'GUIDELINE POSITION:' in answer:
                answer = answer[answer.index('GUIDELINE POSITION:'):]

            break
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            wait_time = (attempt + 1) * 3
            print(f"⚠️ API busy. Retrying in {wait_time}s...")
            time.sleep(wait_time)

    # Smarter conflict detection
    conflict_phrases = [
        "conflict between", "contradicts", "disagrees with",
        "inconsistent with", "in conflict", "directly contradicts",
        "evidence conflicts", "guideline conflict", "conflicting evidence",
        "no consensus", "debate remains", "controversial finding"
    ]
    no_conflict_phrases = [
        "no conflict", "in agreement", "supports the guideline",
        "aligns with", "consistent with the guideline", "no direct conflict",
        "does not conflict"
    ]
    answer_lower = answer.lower()
    has_conflict = (
        any(p in answer_lower for p in conflict_phrases) and
        not any(p in answer_lower for p in no_conflict_phrases)
    )

    # Deduplicate sources by URL
    sources = []
    seen_urls = set()
    for doc, tag in zip(guideline_docs, guideline_citations):
        url = doc.metadata.get("url", "")
        if url not in seen_urls:
            seen_urls.add(url)
            sources.append({
                "citation": tag,
                "type": "guideline",
                "source": doc.metadata.get("source", ""),
                "url": url,
                "section": doc.metadata.get("section", ""),
            })
    for doc, tag in zip(pubmed_docs, pubmed_citations):
        url = doc.metadata.get("url", "")
        if url not in seen_urls:
            seen_urls.add(url)
            sources.append({
                "citation": tag,
                "type": "pubmed",
                "source": doc.metadata.get("source", ""),
                "url": url,
                "section": "",
            })

    return {"answer": answer, "sources": sources, "has_conflict": has_conflict}


print("✅ HIV RAG function ready — powered by Groq (GPT-OSS 20B)")

✅ Key loaded from Colab Secrets
✅ Groq connected: connected
✅ HIV RAG function ready — powered by Groq (GPT-OSS 20B)


In [ ]:
question = 'What is the recommended first-line ART regimen for treatment-naive adults in Nigeria?'
result = hiv_rag(question)
display_result(question, result)

In [ ]:
import time

TEST_QUESTIONS = [
    'What is the recommended first-line ART regimen for treatment-naive adults in Nigeria?',
    'What are the drug interactions between rifampicin and dolutegravir in HIV/TB coinfection?',
    'What does evidence say about long-acting injectable ART compared to daily oral regimens?',
    'What does evidence say about PrEP effectiveness in women?',
    'How should ART be managed during pregnancy?',
    'What prophylaxis is recommended for HIV patients with low CD4 count?',
    'What is the evidence that undetectable viral load prevents HIV transmission?',
    'Is TDF or TAF safer for patients with renal impairment?',
    'How does depression affect ART adherence in people living with HIV?',
]

for q in TEST_QUESTIONS:
    while True:
        try:
            result = hiv_rag(q)
            display_result(q, result)
            break
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                print('⏳ Rate limit. Pausing 20 seconds...')
                time.sleep(20)
            else:
                print('Error:', e)
                break
    time.sleep(8)

print('✅ All questions completed')

✅ All questions completed


In [ ]:
import os, json, shutil, zipfile
from google.colab import files

# Export chunks as JSON — this replaces ChromaDB entirely
data = [{"text": chunks[i], "metadata": metadatas[i]} for i in range(len(chunks))]
print('Total chunks:', len(data))

# Upload the new hiv_streamlit.zip from your computer
print('\nNow upload hiv_streamlit.zip when the picker opens...')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

Total chunks: 1812

Now upload hiv_streamlit.zip when the picker opens...


Saving hiv_streamlit.zip to hiv_streamlit.zip
Uploaded: ['hiv_streamlit.zip']


In [ ]:
import os, json, shutil, zipfile

# Unzip
with zipfile.ZipFile('/content/hiv_streamlit.zip', 'r') as z:
    z.extractall('/content/')
print('✅ Unzipped')

# Write chunks.json into the streamlit folder
data = [{"text": chunks[i], "metadata": metadatas[i]} for i in range(len(chunks))]
with open('/content/hiv_streamlit/chunks.json', 'w') as f:
    json.dump(data, f)
print('✅ chunks.json written —', len(data), 'chunks')

# Confirm structure
print('\nContents:')
for f in os.listdir('/content/hiv_streamlit'):
    print(' ', f)

✅ Unzipped
✅ chunks.json written — 1812 chunks

Contents:
  .streamlit
  requirements.txt
  chunks.json
  app.py


In [ ]:
import os
from google.colab import userdata

os.chdir('/content/hiv_streamlit')

token     = userdata.get('GITHUB_TOKEN')
username  = 'kiks2022'
repo_name = 'HIV-Evidence-Assistant-LLM-RAG'

!git config --global user.name "kiks2022"
!git config --global user.email "adekoyaakorede@gmail.com"
!git init
!git add .
!git commit -m "HIV Evidence Assistant — clean Streamlit no LangChain"
!git branch -M main
!git remote remove origin 2>/dev/null || true
!git remote add origin https://{token}@github.com/{username}/{repo_name}.git
!git push -u origin main --force
print('✅ Pushed to GitHub')

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/hiv_streamlit/.git/
[master (root-commit) 63bda34] HIV Evidence Assistant — clean Streamlit no LangChain
 4 files changed, 350 insertions(+)
 create mode 100644 .streamlit/config.toml
 create mode 100644 app.py
 create mode 100644 chunks.json
 create mode 100644 requirements.txt
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 274.82 KiB | 5.9

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import json

# Load chunks
with open('/content/hiv_streamlit/chunks.json') as f:
    data = json.load(f)

texts = [d['text'] for d in data]

# Embed once
print('Embedding', len(texts), 'chunks...')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

# Save as numpy file
np.save('/content/hiv_streamlit/embeddings.npy', embeddings)
print('✅ Saved embeddings.npy —', embeddings.shape)

Embedding 1812 chunks...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

✅ Saved embeddings.npy — (1812, 384)


In [ ]:
# Open app.py and replace the load_resources function
new_load = '''@st.cache_resource(show_spinner="Loading knowledge base...")
def load_resources():
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    with open("chunks.json") as f:
        data = json.load(f)
    texts     = [d["text"]     for d in data]
    metadatas = [d["metadata"] for d in data]
    embeddings = np.load("embeddings.npy")
    groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
    return model, embeddings, texts, metadatas, groq_client, len(texts)'''

with open('/content/hiv_streamlit/app.py', 'r') as f:
    content = f.read()

# Find and replace the old load_resources function
import re
content = re.sub(
    r'@st\.cache_resource.*?return model, embeddings, texts, metadatas, groq_client, len\(texts\)',
    new_load,
    content,
    flags=re.DOTALL
)

with open('/content/hiv_streamlit/app.py', 'w') as f:
    f.write(content)

print('✅ app.py updated')

✅ app.py updated


In [ ]:
import os
from google.colab import userdata

os.chdir('/content/hiv_streamlit')
token = userdata.get('GITHUB_TOKEN')

!git add embeddings.npy app.py
!git commit -m "Pre-computed embeddings — faster startup no throttling"
!git push origin main
print('✅ Pushed')

[main b8d06be] Pre-computed embeddings — faster startup no throttling
 2 files changed, 2 insertions(+), 16 deletions(-)
 create mode 100644 embeddings.npy
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.46 MiB | 1.92 MiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/kiks2022/HIV-Evidence-Assistant-LLM-RAG.git
   63bda34..b8d06be  main -> main
✅ Pushed


In [ ]:
import os
os.chdir('/content/hiv_streamlit')
from google.colab import userdata

# Fix requirements.txt
with open('requirements.txt', 'w') as f:
    f.write("""streamlit==1.35.0
groq==0.9.0
httpx==0.27.0
sentence-transformers==3.0.1
numpy>=1.24.0
pillow>=10.0.0
""")
print('✅ requirements.txt updated')

token = userdata.get('GITHUB_TOKEN')
!git add requirements.txt
!git commit -m "Fix httpx version conflict with groq"
!git push origin main
print('✅ Pushed')

✅ requirements.txt updated
[main c534bab] Fix httpx version conflict with groq
 1 file changed, 1 insertion(+)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 363 bytes | 363.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/kiks2022/HIV-Evidence-Assistant-LLM-RAG.git
   b8d06be..c534bab  main -> main
✅ Pushed


In [ ]:
import os
os.chdir('/content/hiv_streamlit')

with open('app.py', 'r') as f:
    content = f.read()

# Replace the sidebar examples section
old = '''    for ex in examples:
        label = ex[:58] + "..." if len(ex) > 58 else ex
        if st.button(label, key=ex, use_container_width=True):
            st.session_state["prefill"] = ex
            st.rerun()'''

new = '''    for ex in examples:
        label = ex[:58] + "..." if len(ex) > 58 else ex
        if st.button(label, key=ex, use_container_width=True):
            st.session_state["selected_question"] = ex
            st.rerun()'''

content = content.replace(old, new)

# Replace the prefill line
old2 = '''# Pre-fill from sidebar button
prefill = st.session_state.pop("prefill", "")

question = st.text_area(
    "Clinical Evidence Question",
    value=prefill,
    placeholder="e.g. What are the drug interactions between rifampicin and dolutegravir in HIV/TB coinfection?",
    height=100,
    key="q_input"
)

col1, _ = st.columns([1, 5])
with col1:
    run = st.button("Analyse Evidence", type="primary", use_container_width=True)'''

new2 = '''# Handle selected question from sidebar
selected = st.session_state.get("selected_question", "")

question = st.text_area(
    "Clinical Evidence Question",
    value=selected,
    placeholder="e.g. What are the drug interactions between rifampicin and dolutegravir in HIV/TB coinfection?",
    height=100,
    key="q_input"
)

col1, _ = st.columns([1, 5])
with col1:
    run = st.button("Analyse Evidence", type="primary", use_container_width=True)

# Auto-run when question selected from sidebar
if selected and not run:
    run = True
    st.session_state.pop("selected_question", None)'''

content = content.replace(old2, new2)

with open('app.py', 'w') as f:
    f.write(content)

print('✅ app.py updated')

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git add app.py
!git commit -m "Fix sidebar buttons auto-fill and auto-run"
!git push origin main
print('✅ Pushed')

✅ app.py updated
[main 6babc52] Fix sidebar buttons auto-fill and auto-run
 1 file changed, 9 insertions(+), 4 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 472 bytes | 472.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/kiks2022/HIV-Evidence-Assistant-LLM-RAG.git
   c534bab..6babc52  main -> main
✅ Pushed


In [ ]:
import os
from google.colab import files, userdata

token = userdata.get('GITHUB_TOKEN')

# Step 1 — clone repo
!git clone https://{token}@github.com/kiks2022/HIV-Evidence-Assistant-LLM-RAG.git /content/hiv_streamlit

# Step 2 — change into folder
os.chdir('/content/hiv_streamlit')
print('In folder:', os.getcwd())
print('Files:', os.listdir('.'))

# Step 3 — upload README
print('Upload README.md when picker opens...')
uploaded = files.upload()

# Step 4 — push
!git config --global user.email "adekoyaakorede@gmail.com"
!git config --global user.name "kiks2022"
!git add README.md
!git commit -m "Add README with live demo link"
!git push origin main
print('Done')

Cloning into '/content/hiv_streamlit'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 17 (delta 5), reused 16 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 2.73 MiB | 8.99 MiB/s, done.
Resolving deltas: 100% (5/5), done.
In folder: /content/hiv_streamlit
Files: ['requirements.txt', 'chunks.json', '.git', 'app.py', 'embeddings.npy', '.streamlit']
Upload README.md when picker opens...


Saving README.md to README.md
[main 328397b] Add README with live demo link
 1 file changed, 124 insertions(+)
 create mode 100644 README.md
Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 3.50 KiB | 3.50 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/kiks2022/HIV-Evidence-Assistant-LLM-RAG.git
   6babc52..328397b  main -> main
Done
